## Setup
Criámos um ficheiro .env com o token criado em https://huggingface.co/settings/tokens com permissão *Read*.

In [1]:
! pip install python-dotenv huggingface_hub reportlab


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
'''
gen_report.ipynb

Módulo 3
'''

from dotenv import dotenv_values
from huggingface_hub import InferenceClient
import pandas as pd
import json
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4

c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Criação da Função call_llm

In [3]:


config = dotenv_values("token.env")
token = config["HF_TOKEN"]

# Cria UM cliente partilhado por todas as chamadas neste notebook.
client = InferenceClient(
    model="meta-llama/Meta-Llama-3-8B-Instruct",
    token=token,   # KeyError claro se o .env estiver mal configurado
)


def call_llm(prompt: str, temperature: float = 0.7, max_tokens: int = 800) -> str:
    """Envia uma mensagem ao modelo e devolve apenas o texto da resposta."""
    response = client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return response.choices[0].message["content"]

## Carregamento dos Resultados dos Módulos 1 e 2

In [5]:
avisos = pd.read_csv("air_quality_plus_avisos_e_recomendacoes.csv", sep=";").iloc[49:72] # valores do dia 7/9/25 em Lisboa
bayes = pd.read_csv("bayes_resultados.csv").iloc[1250:1273] #valores do dia 27/9/25 no Porto
metrics = pd.read_csv("metrics.csv")





def gerar_resumo_executivo(contexto: str, dados: str) -> str:
    prompt = f"""
És um assistente de IA Generativa para apoio à decisão em cidades sustentáveis.

Com base no contexto e dados abaixo, escreve um resumo executivo com no máximo 200 palavras,
em português europeu, claro e adequado para decisores municipais e cidadãos.

CONTEXTO:
\"\"\"
{contexto}
\"\"\"

DADOS:
\"\"\"
{dados}
\"\"\"

O resumo deve mencionar:
- principais riscos ambientais;
- desempenho dos modelos;
- utilidade para apoio à decisão.

Devolve apenas o resumo executivo.
"""
    return call_llm(prompt, temperature=0.3, max_tokens=500)

## Criação da Função para Recomendações

In [6]:
def gerar_recomendacoes(contexto: str, dados: str) -> str:
    prompt = f"""
Com base no contexto e dados abaixo, gera 3 recomendações de ação para uma câmara municipal.

CONTEXTO:
\"\"\"
{contexto}
\"\"\"

DADOS:
\"\"\"
{dados}
\"\"\"

REGRAS:
- Português europeu.
- Recomendações curtas e práticas.
- Cada recomendação deve começar por um verbo.
- Foco em qualidade do ar, mobilidade e alertas à população.

FORMATO:
1. ...
2. ...
3. ...

Devolve apenas as recomendações.
"""
    return call_llm(prompt, temperature=0.4, max_tokens=400)

## Criação da Função para Limitações e Riscos

In [7]:
def gerar_limitacoes_riscos(contexto: str , dados: str, modelo: str) -> str:
    prompt = f"""
Analisa criticamente o sistema de IA descrito no contexto , dados  e modelo(s) abaixo.

CONTEXTO:
\"\"\"
{contexto}
\"\"\"

DADOS:
\"\"\"
{dados}
\"\"\"

MODELO USADO:
\"\"\"
{modelo}
\"\"\"

Escreve uma secção chamada "Limitações e riscos", incluindo:
- limitações dos dados;
- risco de enviesamento;
- risco de falsas previsões;
- risco de alucinações da IA generativa;
- necessidade de supervisão humana.

Português europeu.
Devolve apenas a secção.
"""
    return call_llm(prompt, temperature=0.3, max_tokens=600)

## Criação do Relatório

In [9]:
def criador_relatorio(numero, prompt, resumo, recomendacoes, limitacoes):
    relatorio = f"""
    # Relatório Gerado por IA Generativa nº{numero}

    ## Prompt : {prompt}

    ## Resumo Executivo

    {resumo}

    ## Recomendações

    {recomendacoes}

    ## Limitações e Riscos

    {limitacoes}
    """

    return relatorio

def criador_pdf(texto, nome_ficheiro):

    # criar um documento pfd
    doc = SimpleDocTemplate(nome_ficheiro, pagesize=A4)

    #estilo do texto
    estilo = getSampleStyleSheet()

    #elementos do pdf
    elementos = []

    #adicionar o texto linha a linha
    for linha in texto.split("\n"):
        
        # se linha for vazia , adiciona espaço vazio
        if linha.strip()== "":
            elementos.append(Spacer(1,12))
        else:
            elementos.append(Paragraph(linha, estilo["BodyText"]))


    # gerar o pdf      
    doc.build(elementos)

    print(f"Pdf criado comsucesso: {nome_ficheiro}")




# conversão de dados para strings
dados_texto = avisos.to_string() 
dados_texto2 = bayes.to_string()
dados_texto3 = metrics.to_string()

#contexto dos dados 
contexto1 = "Qualidade do ar e avisos metereológicos por hora para o dia 7/9/25 em Lisboa"
contexto2 ="Qualidade do ar e aviso de tempestade por hora para o dia 27/9/25 no Porto"
contexto3 ="Métricas dos modelos usados para previsão de qualidade do Ar (classificação) e da quantidade de Dióxido de Nitrogénio (regressão) "

# modelo(s) usados para obtenção dos dados
modelo1 = "Motor de inferência"
modelo2 = "Rede Bayesiana"
modelo3 = "Logistic Regression e Random Forest (algoritmos de classificação), Linear Regression(algoritmo de regressão)"

# todas as prompts em formato de array
prompts = [[contexto1, dados_texto, modelo1],[contexto2, dados_texto2, modelo2],[contexto3, dados_texto3, modelo3]]

# Array de rellatórios
relatorios = []

for i, prompt in enumerate(prompts):

    #nº do relatório
    numero = i + 1

    #informação present em cada prompt
    contexto = prompt[0]
    dados= prompt[1]
    modelo = prompt[2]

    #geração de cada parte da prompt
    resumo = gerar_resumo_executivo(contexto, dados)
    recomendacoes = gerar_recomendacoes(contexto, dados)
    limitacoes = gerar_limitacoes_riscos(contexto, dados,modelo)

    #criação de um relatório completo
    relatorio = criador_relatorio(numero,contexto, resumo, recomendacoes, limitacoes)

    # criação de um pdf com o relatório
    nome_pdf = f"Relatorio_numero_{numero}.pdf"
    criador_pdf(relatorio, nome_pdf)

    # adição do relatório gerado ao array
    relatorios.append(relatorio)

#criação de prompts_example.md

relatorio_final = f"""
# Exemplos de prompts e output

{relatorios[0]}

{relatorios[1]}

{relatorios[2]}
"""
# geração do ficheiro .md
with open("prompts_examples.md", "w", encoding="utf-8") as f:
    f.write(relatorio_final)

criador_pdf(relatorio_final,"prompts_examples.pdf")

print(relatorio_final)

Pdf criado comsucesso: Relatorio_numero_1.pdf
Pdf criado comsucesso: Relatorio_numero_2.pdf
Pdf criado comsucesso: Relatorio_numero_3.pdf
Pdf criado comsucesso: prompts_examples.pdf

# Exemplos de prompts e output


    # Relatório Gerado por IA Generativa nº1

    ## Prompt : Qualidade do ar e avisos metereológicos por hora para o dia 7/9/25 em Lisboa

    ## Resumo Executivo

    **Resumo Executivo: Qualidade do Ar em Lisboa em 07/09/25**

**Principais Riscos Ambientais:**

A análise dos dados indica que a qualidade do ar em Lisboa em 07/09/25 apresentou níveis elevados de poluentes, incluindo NO2, O3 e PM2.5. Além disso, foram detectados avisos de precipitação em várias horas do dia, o que pode afetar a qualidade do ar e a segurança pública.

**Desempenho dos Modelos:**

Os modelos utilizados para prever a qualidade do ar em Lisboa apresentaram um desempenho razoável, com uma precisão de cerca de 80% em relação aos dados reais. No entanto, é importante notar que os modelos podem ser